### Processing EMTP-RV Parametric Studio outputs

In [ ]:
import pickle
import numpy as np

In [ ]:
from simulation import Simulations

In [ ]:
from utils import create_white_list
from readme import read_me_v0

In [ ]:
# Directory path (EDIT):
simulation_path = ''
# Model file name (without extension):
name = 'IEEE39_Wind_v5_xxx'  # EDIT HERE

In [ ]:
# List of machine variable names:
varnames = [
    '/Teta_1_SM1',   # rotor angle
    '/Omega_1_SM1',  # rotor speed
    '/PowerAng_SM1', # power angle
    '/Pe_SM1',       # electrical power
    '/vd_SM1',  # d-axis stator voltage
    '/id_SM1',  # d-axis stator current
    '/Ef_SM1',  # EMF voltage (q-axis)
    '/vq_SM1',  # q-axis stator voltage
    '/iq_SM1',  # q-axis stator current
]
# List of machines that are excluded.
exclude = []

# List of bus voltages.
buses = np.arange(start=1, stop=30).tolist()
# Buses 30 and 32 to 38 are skipped, since they are
# between the generator and its step-up transformer.
buses.extend([31, 39])  # generator buses with load
# New artificial buses in the middle of each transmission line.
middle_points = np.arange(start=40, stop=73).tolist()
buses.extend(middle_points)

# Power system variant & SC duration.
variant = 'V0'
ibrs = False        # no renewables
sc_time = 'T100ms'  # short-circuit duration

In [ ]:
# Generate the "white_list" variable.
white_list = create_white_list(varnames, exclude, buses, ibrs)
white_list

In [ ]:
# Build all simulations.
sims = Simulations(simulation_path, name, white_list)
sims.build_all_simulations()

In [ ]:
# Dictionary holding DataFrames of signals from all simulations.
nsim = sims.get_nb_simu_tot()
print(f'Total no. of simulations: {nsim}')

# Dict keys for simulations.
sim_keys = ['SC3-' + 'BUS'+str(k) for k in buses]
sim_keys.extend(['SC2-' + 'BUS'+str(k) for k in buses])
sim_keys.extend(['SC1-' + 'BUS'+str(k) for k in buses])

if nsim != len(sim_keys):
    raise ValueError()

# Name pairs for renaming select columns.
name_pairs = {
    # Wind farm signals.
    'FFC_WP2/Wind_Turbine/PMSG_T_rotor': 'WindFarm/PMSG_T_rotor',
    'FFC_WP2/Wind_Turbine/PMSG_w_rotor': 'WindFarm/PMSG_w_rotor',
    'FFC_WP2/Converter_control/Control/Grid_Ctrl/FRT_flag': 'WindFarm/FRT_flag',
    # Solar park signals.
    'WECC_PVPark_1/Converter_control/Control/GridControl_DLL/FRT_flag': 'PVPark/FRT_flag',
}

data = {}
data['README'] = read_me_v0
for key, index in zip(sim_keys, range(nsim)):
    # Export signals to DataFrame.
    sim = sims.get_simulation(index)
    sim_df = sim.to_dataframe()
    if ibrs:
        sim_df.rename(columns=name_pairs, inplace=True)
    # Assign DataFrame to a simulation key.
    data[key] = sim_df

In [ ]:
# Pickle data to the external file.
file_name = variant + '-' + sc_time + '.pkl'
with open(file=file_name, mode='wb') as fp:
    pickle.dump(data, fp)